In [13]:
import os
import pandas as pd

# Ruta al escritorio del usuario
desktop = os.path.join(os.path.expanduser("~"), "Desktop")

# Lista de archivos Parquet y nombres de salida CSV
files = {
    "app/tables/temp_acontecimientos_por_dia.parquet": "temp_acontecimientos_por_dia.csv",
    "app/tables/temp_coctel_fuente_fb.parquet": "temp_coctel_fuente_fb.csv",
    "app/tables/temp_coctel_temas.parquet": "temp_coctel_temas.csv",
    "app/tables/temp_usuarios_7_dias.parquet": "temp_usuarios_7_dias.csv",
    "app/tables/temp_usuarios_por_dia.parquet": "temp_usuarios_por_dia.csv",
    "app/tables/temp_usuarios_ultimo_dia.parquet": "temp_usuarios_ultimo_dia.csv"
}

# Convertir cada archivo Parquet a CSV y guardarlo en el escritorio
for parquet_file, csv_file in files.items():
    try:
        # Leer el archivo Parquet
        df = pd.read_parquet(parquet_file)
        # Crear la ruta completa para guardar el archivo en el escritorio
        csv_path = os.path.join(desktop, csv_file)
        # Guardar como CSV
        df.to_csv(csv_path, index=False)
        print(f"Archivo convertido: {parquet_file} -> {csv_path}")
    except Exception as e:
        print(f"Error al convertir {parquet_file}: {e}")


Archivo convertido: app/tables/temp_acontecimientos_por_dia.parquet -> C:\Users\YOGA\Desktop\temp_acontecimientos_por_dia.csv
Archivo convertido: app/tables/temp_coctel_fuente_fb.parquet -> C:\Users\YOGA\Desktop\temp_coctel_fuente_fb.csv
Archivo convertido: app/tables/temp_coctel_temas.parquet -> C:\Users\YOGA\Desktop\temp_coctel_temas.csv
Archivo convertido: app/tables/temp_usuarios_7_dias.parquet -> C:\Users\YOGA\Desktop\temp_usuarios_7_dias.csv
Archivo convertido: app/tables/temp_usuarios_por_dia.parquet -> C:\Users\YOGA\Desktop\temp_usuarios_por_dia.csv
Archivo convertido: app/tables/temp_usuarios_ultimo_dia.parquet -> C:\Users\YOGA\Desktop\temp_usuarios_ultimo_dia.csv


In [2]:
import pandas as pd
temp_coctel_fuente = pd.read_parquet('app/tables/temp_base_coctel.parquet')
temp_coctel_fuente_programas = temp_coctel_fuente
temp_coctel_fuente_fb = pd.read_parquet('app/tables/temp_coctel_fuente_fb.parquet')
temp_coctel_fuente_actores = pd.read_parquet('app/tables/temp_coctel_fuente_actores.parquet')
temp_coctel_temas = pd.read_parquet('app/tables/temp_coctel_temas.parquet')

lugares_uniques = temp_coctel_fuente['lugar'].unique().tolist()

temp_coctel_fuente['fecha_registro'] = pd.to_datetime(temp_coctel_fuente['fecha_registro'])
temp_coctel_fuente_programas['fecha_registro'] = pd.to_datetime(temp_coctel_fuente_programas['fecha_registro'])
temp_coctel_fuente_fb['fecha_registro'] = pd.to_datetime(temp_coctel_fuente_fb['fecha_registro'])
temp_coctel_fuente_actores['fecha_registro'] = pd.to_datetime(temp_coctel_fuente_actores['fecha_registro'])
temp_coctel_temas['fecha_registro'] = pd.to_datetime(temp_coctel_temas['fecha_registro'])

temp_coctel_fuente['coctel'] = pd.to_numeric(temp_coctel_fuente['coctel'], errors='coerce')
temp_coctel_fuente_programas['coctel'] = pd.to_numeric(temp_coctel_fuente_programas['coctel'], errors='coerce')
temp_coctel_fuente_fb['coctel'] = pd.to_numeric(temp_coctel_fuente_fb['coctel'], errors='coerce')
temp_coctel_fuente_actores['coctel'] = pd.to_numeric(temp_coctel_fuente_actores['coctel'], errors='coerce')
temp_coctel_temas['coctel'] = pd.to_numeric(temp_coctel_temas['coctel'], errors='coerce')

temp_coctel_fuente['coctel']=temp_coctel_fuente['coctel'].fillna(0.0)
temp_coctel_fuente_programas['coctel']=temp_coctel_fuente_programas['coctel'].fillna(0.0)
temp_coctel_fuente_fb['coctel']=temp_coctel_fuente_fb['coctel'].fillna(0.0)
temp_coctel_fuente_actores['coctel']=temp_coctel_fuente_actores['coctel'].fillna(0.0)
temp_coctel_temas['coctel']=temp_coctel_temas['coctel'].fillna(0.0)

temp_coctel_fuente_programas['nombre_canal'] = temp_coctel_fuente_programas['canal_nombre'] 
temp_coctel_fuente_fb['nombre_canal'] = temp_coctel_fuente_fb['canal_nombre'] 

In [14]:
fecha_inicio_g1="2024-01-08"
fecha_fin_g1="2024-12-06"
fecha_inicio_g1 = pd.to_datetime(fecha_inicio_g1,format='%Y-%m-%d')
fecha_fin_g1 = pd.to_datetime(fecha_fin_g1,format='%Y-%m-%d')
option_lugar_g1='Arequipa'
option_fuente_g1 = 'Radio'
temp_g1 = temp_coctel_fuente[(temp_coctel_fuente['fecha_registro']>=fecha_inicio_g1)&(temp_coctel_fuente['fecha_registro']<=fecha_fin_g1)&
                        (temp_coctel_fuente['lugar']==option_lugar_g1)]
if option_fuente_g1 == 'Radio':
    temp_g1 = temp_g1[temp_g1['id_fuente']==1]
elif option_fuente_g1 == 'TV':
    temp_g1 = temp_g1[temp_g1['id_fuente']==2]
elif option_fuente_g1 == 'Redes':
    temp_g1 = temp_g1[temp_g1['id_fuente']==3]
temp_g1 = temp_g1.drop_duplicates(subset=['id', 'fecha_registro'])
temp_g1['semana'] = temp_g1['fecha_registro'].dt.isocalendar().year.map(str) +'-'+temp_g1['fecha_registro'].dt.isocalendar().week.map(str)
temp_g1 = temp_g1.groupby('semana').agg({'id':'count','coctel':'sum','fecha_registro':'first'}).reset_index()
temp_g1 = temp_g1.rename(columns={'id':'Cantidad'})
temp_g1['porcentaje'] = temp_g1['coctel'] / temp_g1['Cantidad']
temp_g1 = temp_g1.sort_values('fecha_registro')
temp_fecha = pd.DataFrame()
temp_fecha['fecha'] = pd.date_range(start=fecha_inicio_g1, end=fecha_fin_g1)
temp_fecha['semana'] = temp_fecha['fecha'].dt.isocalendar().year.map(str) +'-'+temp_fecha['fecha'].dt.isocalendar().week.map(str)
temp_fecha = temp_fecha.groupby('semana').agg({'fecha':'first'}).reset_index()
temp_fecha = temp_fecha.sort_values('fecha')
del temp_fecha['fecha']
temp_g1 = pd.merge(temp_fecha, temp_g1, how='left', on='semana')
temp_g1['Cantidad'] = temp_g1['Cantidad'].fillna(0)
temp_g1['porcentaje'] = temp_g1['porcentaje'].fillna(0)
temp_g1 = temp_g1.sort_values('fecha_registro')
temp_g1['semana'] = temp_g1['semana'].map(str)
temp_g1["porcentaje"] = temp_g1["porcentaje"] * 100 # a dos decimales
temp_g1["porcentaje"] = temp_g1["porcentaje"].map('{:.2f}'.format)

In [16]:
id_posicion_dict = {1: 'a favor',
                    2: 'potencialmente a favor',
                    3: 'neutral',
                    4: 'potencialmente en contra',
                    5: 'en contra'}
fecha_inicio_g23="2024-01-08"
fecha_fin_g23="2024-12-06"
fecha_inicio_g23 = pd.to_datetime(fecha_inicio_g1,format='%Y-%m-%d')
fecha_fin_g23 = pd.to_datetime(fecha_fin_g1,format='%Y-%m-%d')
option_lugar_g23='Arequipa'
option_fuente_g23 = 'Radio'


temp_g23 = temp_coctel_temas[(temp_coctel_temas['fecha_registro']>=fecha_inicio_g23) &
                                (temp_coctel_temas['fecha_registro']<=fecha_fin_g23) &
                                (temp_coctel_temas['lugar']==option_lugar_g23)]
option_nota_g23 = 'Con coctel'
if option_nota_g23 == 'Con coctel':
    temp_g23 = temp_g23[temp_g23['coctel'] == 1]
    titulo = f"Recuento de mensajes emitidos con coctel por {option_fuente_g23} en {option_lugar_g23} entre {fecha_inicio_g23.strftime('%d-%m-%Y')} y {fecha_fin_g23.strftime('%d-%m-%Y')}"
elif option_nota_g23 == 'Sin coctel':
    temp_g23 = temp_g23[temp_g23['coctel'] == 0]
    titulo = f"Recuento de mensajes emitidos sin coctel por {option_fuente_g23} en {option_lugar_g23} entre {fecha_inicio_g23.strftime('%d-%m-%Y')} y {fecha_fin_g23.strftime('%d-%m-%Y')}"
else:
    titulo = f"Recuento de mensajes emitidos por {option_fuente_g23} en {option_lugar_g23} entre {fecha_inicio_g23.strftime('%d-%m-%Y')} y {fecha_fin_g23.strftime('%d-%m-%Y')}"

if option_fuente_g23 == 'Radio':
    temp_g23 = temp_g23[temp_g23['id_fuente']==1]
elif option_fuente_g23 == 'TV':
    temp_g23 = temp_g23[temp_g23['id_fuente']==2]
elif option_fuente_g23 == 'Redes':
    temp_g23 = temp_g23[temp_g23['id_fuente']==3]

if not temp_g23.empty:
    temp_g23["id_posicion"] = temp_g23["id_posicion"].map(id_posicion_dict)

    # df_grouped = temp_g23.groupby(['descripcion', 'id_posicion']).size().reset_index(name='frecuencia')

    # top_10_temas = df_grouped.groupby('descripcion')['frecuencia'].sum().nlargest(10).index
    # df_top_10 = df_grouped[df_grouped['descripcion'].isin(top_10_temas)]

In [18]:
fecha_inicio_g27="2024-01-08"
fecha_fin_g27="2024-12-06"
fecha_inicio_g27 = pd.to_datetime(fecha_inicio_g1,format='%Y-%m-%d')
fecha_fin_g27 = pd.to_datetime(fecha_fin_g1,format='%Y-%m-%d')
option_lugar_g27='Arequipa'
option_fuente_g27 = 'Radio'
option_nota_g27 = 'Con coctel'

temp_g27 = temp_coctel_fuente_actores[(temp_coctel_fuente_actores['fecha_registro']>=fecha_inicio_g27) &
                                (temp_coctel_fuente_actores['fecha_registro']<=fecha_fin_g27) &
                                (temp_coctel_fuente_actores['lugar']==option_lugar_g27)]

if option_fuente_g27 == 'Radio':
    temp_g27 = temp_g27[temp_g27['id_fuente']==1]
elif option_fuente_g27 == 'TV':
    temp_g27 = temp_g27[temp_g27['id_fuente']==2]
elif option_fuente_g27 == 'Redes':
    temp_g27 = temp_g27[temp_g27['id_fuente']==3]

if option_nota_g27 == 'Con coctel':
    temp_g27 = temp_g27[temp_g27['coctel'] == 1]
    titulo = f"Recuento de posiciones emitidas por actor en {option_lugar_g27} entre {fecha_inicio_g27.strftime('%d-%m-%Y')} y {fecha_fin_g27.strftime('%d-%m-%Y')}. Notas con coctel"
elif option_nota_g27 == 'Sin coctel':
    temp_g27 = temp_g27[temp_g27['coctel'] == 0]
    titulo = f"Recuento de posiciones emitidas por actor en {option_lugar_g27} entre {fecha_inicio_g27.strftime('%d-%m-%Y')} y {fecha_fin_g27.strftime('%d-%m-%Y')}. Notas sin coctel"

else:
    titulo = f"Recuento de posiciones emitidas por actor en {option_lugar_g27} entre {fecha_inicio_g27.strftime('%d-%m-%Y')} y {fecha_fin_g27.strftime('%d-%m-%Y')}"



In [19]:
temp_g27

,id,fecha_registro,acontecimiento,coctel,id_posicion,lugar,color,id_fuente,fuente_nombre,id_canal,canal_nombre,actor_nombre
71,24194,2024-01-12,Manifestó que el Gobierno Regional de Arequipa...,1.0,2,Arequipa,Celeste,1.0,RADIO,90.0,RADIO MELODÍA 104.3 (FM) 1220 (AM),Periodista
72,24195,2024-01-12,Manifestó que destinan 2 millones de soles par...,1.0,2,Arequipa,Celeste,1.0,RADIO,119.0,RADIO VICTORIA 92.9 (FM) 1470 (AM),marco antonio anco
77,24210,2024-01-16,Manifestó que ComexPerú advierte la importanci...,1.0,2,Arequipa,Celeste,1.0,RADIO,131.0,RADIO YARAVÍ 106.9 (FM) 930 (AM),Periodista
78,24210,2024-01-16,Manifestó que ComexPerú advierte la importanci...,1.0,2,Arequipa,Celeste,1.0,RADIO,86.0,RADIO SAN MARTÍN 97.7 (FM) 1380 (AM),Periodista
79,24210,2024-01-16,Manifestó que ComexPerú advierte la importanci...,1.0,2,Arequipa,Celeste,1.0,RADIO,90.0,RADIO MELODÍA 104.3 (FM) 1220 (AM),Periodista
...,...,...,...,...,...,...,...,...,...,...,...,...
250497,25428,2024-06-05,Manifestó que los representantes de 19 regione...,1.0,3,Arequipa,Gris,1.0,RADIO,90.0,RADIO MELODÍA 104.3 (FM) 1220 (AM),Periodista
250498,25428,2024-06-05,Manifestó que los representantes de 19 regione...,1.0,3,Arequipa,Gris,1.0,RADIO,17.0,RADIO LÍDER 98.7 (FM) 1240 (AM),Periodista
250499,25428,2024-06-05,Manifestó que los representantes de 19 regione...,1.0,3,Arequipa,Gris,1.0,RADIO,34.0,RADIO CONTACTO SUR 93.9 (FM) 800 (AM),Periodista
250507,25476,2024-06-12,"Refiere que Corpac, siendo responsable del con...",1.0,3,Arequipa,Gris,1.0,RADIO,119.0,RADIO VICTORIA 92.9 (FM) 1470 (AM),jessica luna
